# Apuntes Fase 3 - Semana 3

Ejemplos del apunte: Datos Preparación

## Exploración inicial
La exploración inicial del dataset raw con pandas

In [1]:
import pandas as pd
import numpy as np

#Cargamos los datos
df = pd.read_csv('../data/raw/predict_students_dropout_and_academic_success.csv', sep=(';'))

# Información general
print('=' * 70)
print(" INFORMACIÓN GENERAL ")
print("=" * 70)
print(f" Forma : {df.shape}")
print(f"\nTipos de datos :")
print(df.dtypes)
print(f"\nPrimeras filas :")
print(df.head())
# Información detallada
print(f"\nInfo :")
df.info()
# Estadísticas descriptivas
print(df.describe())
# Valores faltantes
print("\nVALORES FALTANTES ")
print(df.isnull().sum())
print(f"\nTotal de faltantes : {df.isnull().sum().sum()}")
print(f" Porcentaje : {df.isnull().sum().sum() / df.size * 100:.2f}%")
# Duplicados
print(f" Filas duplicadas : {df.duplicated().sum()}")

 INFORMACIÓN GENERAL 
 Forma : (4424, 37)

Tipos de datos :
Marital status                                      int64
Application mode                                    int64
Application order                                   int64
Course                                              int64
Daytime/evening attendance\t                        int64
Previous qualification                              int64
Previous qualification (grade)                    float64
Nacionality                                         int64
Mother's qualification                              int64
Father's qualification                              int64
Mother's occupation                                 int64
Father's occupation                                 int64
Admission grade                                   float64
Displaced                                           int64
Educational special needs                           int64
Debtor                                              int64
Tuition fees

## Detección de valores atípicos
Indentificación de valores fuera de rango

In [2]:
print("DETECCIÓN DE VALORES ATÍPICOS")

#Edad
if 'Age at enrollment' in df.columns:
    print(f"\nEdad - Min: {df['Age at enrollment'].min()}, Max: {df['Age at enrollment'].max()}")
    edades_negativas = df['Age at enrollment'] < 0
    edades_altas = df['Age at enrollment'] > 120
    if edades_negativas.any():
        print(f" {edades_negativas.sum()} edades negativas")
    if edades_altas.any():
        print(f" {edades_altas.sum()} edades > 120 años")
    else:
        print("NO HAY EDADES ATÍPICAS")

# Addimision grade (between 0 and 200)
if 'Admission grade' in df.columns:
    print(f"\nNota de admisión - Min: {df['Admission grade'].min()}, Max: {df['Admission grade'].max()}")
    notas_negativas = df['Admission grade'] < 0
    notas_sobre_200 = df['Admission grade'] > 200
    if notas_negativas.any():
        print(f" {notas_negativas.sum()} Notas negativas")
    if notas_sobre_200.any():
        print(f" {notas_sobre_200.sum()} Nota > 200")
    else:
        print("NO HAY NOTAS DE ADMISIÓN ATÍPICAS")

DETECCIÓN DE VALORES ATÍPICOS

Edad - Min: 17, Max: 70
NO HAY EDADES ATÍPICAS

Nota de admisión - Min: 95.0, Max: 190.0
NO HAY NOTAS DE ADMISIÓN ATÍPICAS


## Corrección de Problemas
Estrategias de correción:

### 1. Eliminar filas problemáticas:


In [3]:
#Eliminar edades atípicas
df_clean = df[(df['Age at enrollment'] >= 0 ) & (df['Age at enrollment'] <=120)].copy()

#Eliminar filas con demasiados faltantes
threshold = 0.5
df_clean = df_clean.dropna(thresh=int(threshold * df_clean.shape[1]))

### 2.Corregir valores específicos


In [4]:
# corregir edades negativas
df_clean.loc[df_clean['Age at enrollment'] < 0, 'Age at enrollment'] = df_clean['Age at enrollment'].abs()

# Corregir valores por defecto incorrectos
df_clean.loc[df_clean['Admission grade'] == 201, 'Admission grade'] = np.nan

### 3. Estandarizar categorías

In [5]:
# Uniformar Texto
# De las pocas variables categoricas
df_clean.head()
df_clean['Target'] = df_clean['Target'].str.lower().str.strip()
df_clean.head()
# No es neceasrio hacer un mapeo de variantes para esta categoria

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,graduate


### 4. Eliminar duplicados

In [6]:
# Eliminar filas completamente duplicadas

df_clean = df_clean.drop_duplicates()

# Eliminar duplicadas basados en columnas clave, en nuestro caso no hay columnas como ids o algo que sirva para este tipo de limpieza.
#df_clean = df_clean.drop_duplicates(subset=['columna'], keep='first')